# Práctica: Apache Spark Standalone (PySpark + RDDs)

**Materia:** Manejo Masivo de Datos (Big Data) — Prof. Javier Moya
**Alumno:** Andrés Tadeo Carmona Capula — **Matrícula:** 125052403 — **Grupo:** IDIA222
**Fecha:** 18/11/2026

**Entorno usado:** Opción 2 — Docker con Jupyter + PySpark (`jupyter/pyspark-notebook`) en Windows con Docker Desktop.
- Jupyter: http://localhost:8888
- Spark UI: http://localhost:4040 (solo mientras haya una SparkSession activa)

## Teoría: arquitectura de Spark (sección 1 del Notion)
- **Driver:** el programa Python (este notebook) con la lógica; crea la `SparkSession` y divide el trabajo en tareas.
- **Cluster Manager:** asigna CPU y RAM. Aquí es el modo **local** (`local[*]`); en un clúster sería Standalone, YARN, Kubernetes o Mesos.
- **Executors:** procesos JVM que ejecutan las tareas en paralelo y devuelven resultados al Driver.
- **Diferencia con MapReduce:** MapReduce escribe resultados intermedios en disco; Spark los mantiene en RAM, por eso es más rápido.

| Modo | Descripción | Cuándo usar |
|---|---|---|
| Local | Todo en una sola máquina | Desarrollo, pruebas, aprendizaje (**esta práctica**) |
| Standalone | Clúster Spark sin dependencias | Clústeres dedicados a Spark |
| YARN | Integrado con Hadoop | Cuando ya tienes Hadoop |
| Kubernetes | Contenedores orquestados | Entornos cloud-native |
| Mesos | Gestor genérico | Clústeres compartidos |

**SparkSession vs SparkContext:** `SparkSession` (`spark`, Spark 2.0+) es la API unificada (RDDs + DataFrames + SQL) y es la que se usa siempre; para funciones de bajo nivel como `parallelize` se obtiene el contexto con `sc = spark.sparkContext`.

In [1]:
# Verificar la instalación de PySpark dentro del contenedor
import pyspark
print(pyspark.__version__)

3.5.0


## 0. Preparación: descargar los datasets
Se crean las carpetas `DATOS` y `RESULTADOS` y se descargan los 3 archivos que usa la práctica.
> Nota: si vuelves a ejecutar esta celda, `wget` crea copias `archivo.csv.1`; no pasa nada, pero ejecútala solo una vez.

In [2]:
!mkdir -p DATOS
!mkdir -p RESULTADOS
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/pdi_sales_small.csv" -P ./DATOS
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv" -P ./DATOS
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt" -P ./DATOS
!ls -la ./DATOS

--2026-11-19 04:27:25--  https://github.com/josepgarcia/datos/raw/refs/heads/main/pdi_sales_small.csv
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/pdi_sales_small.csv [following]
--2026-11-19 04:27:26--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/pdi_sales_small.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5549121 (5.3M) [text/plain]
Saving to: ‘./DATOS/pdi_sales_small.csv’

pdi_sales_small.csv 100%[===================>]   5.29M  6.99MB/s    in 0.8s    

2026-11-19 04:27:27 (6.99 MB/s) - ‘./DATOS/pdi_sales_small.csv’ saved [554

In [3]:
!head ./DATOS/pdi_sales_small.csv
!head ./DATOS/persons.csv
!head ./DATOS/stocks.txt

ProductID;Date;Zip;Units;Revenue;Country
725;1/15/1999;41540          ;1;115.5;Germany
787;6/6/2002;41540          ;1;314.9;Germany
788;6/6/2002;41540          ;1;314.9;Germany
940;1/15/1999;22587          ;1;687.7;Germany
396;1/15/1999;22587          ;1;857.1;Germany
734;4/10/2003;22587          ;1;330.7;Germany
769;2/15/1999;22587          ;1;257.2;Germany
499;1/15/1999;12555          ;1;846.3;Germany
2254;1/15/1999;40217          ;1;57.7;Germany
name,age,gender,salary
John Doe,30,Male,50000
Jane Smith,25,Female,45000
David Johnson,35,Male,60000
Emily Davis,28,Female,52000
Michael Wilson,40,Male,75000
Sarah Brown,32,Female,58000
Robert Lee,29,Male,51000
Lisa Garcia,27,Female,49000
James Martinez,38,Male,70000
id,name,category,quantity,price
1,iPhone,Electronics,10,899.99
2,Macbook,Electronics,5,1299.99
3,iPad,Electronics,15,499.99
4,Samsung TV,Electronics,8,799.99
5,LG TV,Electronics,10,699.99
6,Nike Shoes,Clothing,30,99.99
7,Adidas Shoes,Clothing,25,89.99
8,Sony Headphones,Electroni

## 1. Crear una SparkSession (versión básica)
`getOrCreate()` crea la sesión si no existe o devuelve la que ya está activa.

In [4]:
from pyspark.sql import SparkSession

# Crear sesión (usa configuración por defecto)
spark = SparkSession.builder.getOrCreate()

# Verificar que funciona
spark

**Salida esperada:** `SparkSession - in-memory · SparkContext · Version: v3.5.0 · Master: local[*] · AppName: pyspark-shell`
- `local[*]`: modo local usando todos los cores disponibles
- `pyspark-shell`: nombre por defecto de la aplicación

### Versión con configuración personalizada
> `getOrCreate()` reutiliza la sesión activa, así que primero se detiene para que la configuración se aplique.
> (En Jupyter, `spark.driver.memory` queda registrada pero solo tiene efecto real si se fija antes de arrancar la JVM.)

In [5]:
SparkSession.builder.getOrCreate().stop()

spark = (SparkSession.builder
    .appName("MiAnalisisDatos")                      # Nombre que aparece en Spark UI
    .config("spark.executor.memory", "2g")           # RAM por executor
    .config("spark.driver.memory", "1g")             # RAM para el driver
    .config("spark.sql.shuffle.partitions", "10")    # Particiones para shuffle
    .getOrCreate())

conf = spark.sparkContext.getConf()
print("AppName              :", spark.sparkContext.appName)
print("spark.executor.memory:", conf.get("spark.executor.memory"))
print("spark.driver.memory  :", conf.get("spark.driver.memory"))
print("shuffle.partitions   :", spark.conf.get("spark.sql.shuffle.partitions"))

# Tip de optimización local: el shuffle se puede cambiar en caliente
spark.conf.set("spark.sql.shuffle.partitions", "10")
print("shuffle (tras conf.set):", spark.conf.get("spark.sql.shuffle.partitions"))

# Se cierra para que el siguiente ejemplo cree su propia sesión "HolaSpark"
spark.stop()

AppName              : MiAnalisisDatos
spark.executor.memory: 2g
spark.driver.memory  : 1g
shuffle.partitions   : 10
shuffle (tras conf.set): 10


## 2. Primer programa: "Hola Spark"
Suma de los números del 0 al 100 con un RDD.

In [6]:
from pyspark.sql import SparkSession

# 1. Crear SparkSession
spark = SparkSession.builder.appName("HolaSpark").getOrCreate()

# 2. Obtener el SparkContext (para funciones de bajo nivel)
sc = spark.sparkContext

# 3. Crear un RDD (Resilient Distributed Dataset)
# parallelize() distribuye los datos en particiones
numeros = sc.parallelize(range(101))

# 4. Realizar una acción: sumar todos los números
total = numeros.sum()
print(f"La suma de 0 a 100 es: {total}")

# 5. Verificar cuántas particiones se crearon
print(f"Número de particiones: {numeros.getNumPartitions()}")

La suma de 0 a 100 es: 5050
Número de particiones: 6


### Lazy evaluation: `map` no se ejecuta hasta que llamamos a una acción (`collect`)

In [7]:
# Esto NO se ejecuta todavía (es una transformación)
numeros_dobles = numeros.map(lambda x: x * 2)

# AHORA se ejecuta todo (es una acción)
resultado = numeros_dobles.collect()
print(resultado[:10])

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


## 3. Spark UI – ejemplo de análisis
Después de ejecutar la celda, abre **http://localhost:4040** y toma capturas de **Jobs**, **Stages** (verás 10 tasks) y **Executors**.

In [8]:
# Crear un RDD grande
rdd_grande = sc.parallelize(range(1000000), 10)

# Transformación + Acción
resultado = rdd_grande.filter(lambda x: x % 2 == 0).count()
print(resultado)
# Ahora ve a http://localhost:4040
# Verás:
# - 1 Job (count)
# - 1 Stage (filter + count)
# - 10 Tasks (una por partición)

500000


## 4. Lectura de archivos: ejemplo de análisis de ventas (`pdi_sales_small.csv`)
En local hay que usar **ruta absoluta** con el prefijo `file://`, si no Spark busca en HDFS.

In [9]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AnalisisVentas").getOrCreate()
sc = spark.sparkContext

# Construir ruta absoluta
path_relativo = "./DATOS/pdi_sales_small.csv"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto
print(f"Leyendo desde: {path_final}")

# Leer archivo (cada línea es un string)
rdd = sc.textFile(path_final)

# Ver las primeras 3 líneas
print(rdd.take(3))

Leyendo desde: file:///home/jovyan/work/projects/DATOS/pdi_sales_small.csv
['ProductID;Date;Zip;Units;Revenue;Country', '725;1/15/1999;41540          ;1;115.5;Germany', '787;6/6/2002;41540          ;1;314.9;Germany']


### Paso 3: extraer país y unidades (versión ineficiente, del Notion)
⚠️ Hace `split(";")` **dos veces** por línea; en el siguiente paso se optimiza.

In [10]:
# Extraer (País, Unidades) de cada línea
# Formato: ProductID;Date;Zip;Units;Revenue;Country
#              0      1    2    3      4       5
paises_unidades = rdd.map(lambda linea: (
    linea.split(";")[-1].strip(),   # País (última columna)
    linea.split(";")[3]             # Unidades (columna 3)
))
paises_unidades.take(5)

[('Country', 'Units'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1')]

### Pasos 4 y 5: optimización (split una sola vez) y eliminar encabezado

In [11]:
# Paso 4 – ✅ MEJOR: Función que hace split una sola vez
def parsear_linea(linea):
    columnas = linea.split(";")
    pais = columnas[-1].strip()
    unidades = columnas[3]
    return (pais, unidades)

paises_unidades = rdd.map(parsear_linea)

# Obtener la primera línea (encabezado)
header = paises_unidades.first()
print(f"Encabezado: {header}")

# Filtrar todas las líneas que NO sean el encabezado
paises_unidades_sin_header = paises_unidades.filter(lambda linea: linea != header)
paises_unidades_sin_header.take(5)

Encabezado: ('Country', 'Units')


[('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1'),
 ('Germany', '1')]

In [12]:
# Convertir unidades a enteros
paises_unidades_int = paises_unidades_sin_header.map(
    lambda x: (x[0], int(x[1]))
)

# Sumar unidades por país usando reduceByKey
paises_total_unidades = paises_unidades_int.reduceByKey(lambda a, b: a + b)

# Obtener resultados
resultados = paises_total_unidades.collect()

# Mostrar top 10 países por unidades vendidas
top10 = sorted(resultados, key=lambda x: x[1], reverse=True)[:10]
for pais, unidades in top10:
    print(f"{pais}: {unidades:,} unidades")

Germany: 31,746 unidades
France: 31,739 unidades
Canada: 31,148 unidades
Mexico: 31,095 unidades


In [13]:
# Versión con DataFrame (más simple y rápido)
df = spark.read.csv(path_final, header=True, sep=";", inferSchema=True)
df.groupBy("Country").sum("Units").show()

+-------+----------+
|Country|sum(Units)|
+-------+----------+
|Germany|     31746|
|France |     31738|
|Canada |     31148|
|Mexico |     31095|
| France|         1|
+-------+----------+



## 5. Conexión con HDFS (sección 7 del Notion) — solo referencia
Esta sección **solo funciona con el entorno BDA Lab (Opción 1)**, que trae Hadoop. En la Opción 2 (este contenedor) no hay HDFS,
así que no se ejecuta; se deja documentada la diferencia de rutas:

```python
# Archivo LOCAL (tu máquina)
rdd_local = sc.textFile("file:///ruta/absoluta/archivo.csv")
# Archivo en HDFS (sistema distribuido)
rdd_hdfs = sc.textFile("/user/hadoop/datos/archivo.csv")
rdd_hdfs = sc.textFile("hdfs://localhost:9000/user/hadoop/datos/archivo.csv")

# Flujo completo en BDA Lab:
# hdfs dfs -put ./DATOS/pdi_sales_small.csv /user/hadoop/datos/        (terminal)
# rdd = sc.textFile("/user/hadoop/datos/pdi_sales_small.csv")
# paises_unidades.saveAsTextFile("/user/hadoop/resultados/ventas_por_pais")
# hdfs dfs -cat /user/hadoop/resultados/ventas_por_pais/part-00000     (terminal)
```
**Ventaja de HDFS:** los datos están distribuidos y replicados; si un nodo falla, Spark lee otra réplica.

## 6. Finalizar sesión y errores comunes (sección 8 del Notion)
| Error | Causa | Solución |
|---|---|---|
| `JAVA_HOME is not set` | Spark no encuentra Java | No aplica en Docker (la imagen ya trae Java) |
| `FileNotFoundException` | Ruta mal escrita o sin `file://` | `"file://" + os.path.abspath(ruta)` |
| `Port 4040 is already in use` | Otra sesión abierta | `spark.stop()` o `.config("spark.ui.port", "4041")` |
| `Py4JJavaError` en `collect()` | Demasiados datos al driver | `take(n)` o `saveAsTextFile` |

La sesión se cierra con `spark.stop()` (se hace en el Ejercicio 7 y al final del proyecto).

---
# EJERCICIOS PRÁCTICOS

## Ejercicio 1: SparkSession personalizada
**Importante:** `getOrCreate()` devuelve la sesión que ya existe y **no aplica** la nueva configuración.
Por eso primero detenemos la sesión anterior.

In [14]:
from pyspark.sql import SparkSession

# Cerrar la sesión previa para que la nueva configuración sí se aplique
SparkSession.builder.getOrCreate().stop()

spark = (SparkSession.builder
         .appName("MiPrimerAnalisis")                     # 1. nombre
         .config("spark.executor.memory", "1g")           # 2. 1GB por executor
         .config("spark.sql.shuffle.partitions", "8")     # 3. 8 particiones de shuffle
         .getOrCreate())
sc = spark.sparkContext

# 4. Verificar la sesión
print("Nombre de la app      :", sc.appName)
print("Versión de Spark      :", spark.version)
print("Master                :", sc.master)
print("spark.executor.memory :", sc.getConf().get("spark.executor.memory"))
print("shuffle.partitions    :", spark.conf.get("spark.sql.shuffle.partitions"))
spark

Nombre de la app      : MiPrimerAnalisis
Versión de Spark      : 3.5.0
Master                : local[*]
spark.executor.memory : 1g
shuffle.partitions    : 8


## Ejercicio 2: Operaciones básicas con RDDs

In [15]:
numeros = list(range(1, 21))  # [1, 2, 3, ..., 20]

rdd_num = sc.parallelize(numeros)                       # 1. Crear RDD
print("Total de elementos:", rdd_num.count())           # 2. count
print("Primeros 5:", rdd_num.take(5))                   # 3. take
print("Suma:", rdd_num.sum())                           # 4. sum
print(f"Máximo: {rdd_num.max()}, Mínimo: {rdd_num.min()}")  # 5. max / min

Total de elementos: 20
Primeros 5: [1, 2, 3, 4, 5]
Suma: 210
Máximo: 20, Mínimo: 1


## Ejercicio 3: Transformaciones `map` y `filter`

In [16]:
numeros = sc.parallelize(range(1, 51))  # 1 a 50

triples = numeros.map(lambda x: x * 3)             # 1. multiplicar por 3
mayores_50 = triples.filter(lambda x: x > 50)      # 2. filtrar > 50

print("Números después de multiplicar por 3:", triples.collect())
print("Números mayores a 50:", mayores_50.collect())
print("Total de números mayores a 50:", mayores_50.count())   # 3. contar
print("Primeros 10:", mayores_50.take(10))                    # 4. primeros 10

Números después de multiplicar por 3: [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150]
Números mayores a 50: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150]
Total de números mayores a 50: 34
Primeros 10: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78]


## Ejercicio 4: Pares clave-valor

In [17]:
ventas = [
    ("Laptop", 1200),
    ("Mouse", 25),
    ("Laptop", 1100),
    ("Teclado", 75),
    ("Mouse", 30),
    ("Laptop", 1300)
]

rdd_ventas = sc.parallelize(ventas)                                   # 1. RDD

total_por_producto = rdd_ventas.reduceByKey(lambda a, b: a + b)       # 2. total por producto
print("Total por producto:", sorted(total_por_producto.collect()))

num_ventas = rdd_ventas.mapValues(lambda v: 1).reduceByKey(lambda a, b: a + b)  # 3. nº de ventas
print("Número de ventas:", sorted(num_ventas.collect()))

mayor = total_por_producto.sortBy(lambda x: x[1], ascending=False).first()     # 4. mayor total
print(f"Producto con mayor total: {mayor[0]} ({mayor[1]})")

Total por producto: [('Laptop', 3600), ('Mouse', 55), ('Teclado', 75)]
Número de ventas: [('Laptop', 3), ('Mouse', 2), ('Teclado', 1)]
Producto con mayor total: Laptop (3600)


## Ejercicio 5: Lectura y procesamiento de `persons.csv`
> **Nota:** el archivo real del repositorio tiene las columnas `name,age,gender,salary`
> (no `name,age,city` como dice el enunciado), por eso el resultado real son **10 personas** mayores de 30.

In [18]:
import os

# 2. Cargar con sc.textFile y ruta correcta (absoluta + file://)
path_personas = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd_personas = sc.textFile(path_personas)
print("Archivo cargado:", rdd_personas.take(3))

# 3. Eliminar encabezado
header_personas = rdd_personas.first()
personas_sin_header = rdd_personas.filter(lambda linea: linea != header_personas)
print("Encabezado eliminado:", personas_sin_header.take(2))

# 4. Extraer nombre y edad (split una sola vez)
def parsear_persona(linea):
    columnas = linea.split(",")
    return (columnas[0], int(columnas[1]))

nombres_edades = personas_sin_header.map(parsear_persona)

# 5. Filtrar mayores de 30
mayores_30 = nombres_edades.filter(lambda x: x[1] > 30)
for nombre, edad in mayores_30.collect():
    print(f"{nombre} ({edad})")

# 6. Contar
print("Total:", mayores_30.count(), "personas")

Archivo cargado: ['name,age,gender,salary', 'John Doe,30,Male,50000', 'Jane Smith,25,Female,45000']
Encabezado eliminado: ['John Doe,30,Male,50000', 'Jane Smith,25,Female,45000']
David Johnson (35)
Michael Wilson (40)
Sarah Brown (32)
James Martinez (38)
William Anderson (33)
Karen Hernandez (31)
Christopher Taylor (37)
Matthew Davis (36)
Daniel Miller (34)
Linda Martin (39)
Total: 10 personas


## Ejercicio 6: Optimización – evitar múltiples `split()`
**1. Problema:** el código original hace `split(",")` **3 veces por línea** (una por cada `map`) y además recorre el RDD 3 veces.
Con 1 millón de líneas son 3 millones de `split` y 3 lecturas del archivo.

In [19]:
# 2 y 3. Versión optimizada: un solo split y una sola pasada
# (se aplica a persons.csv: name, age, gender -> la 3ª columna es 'gender' en vez de 'city')
def parsear(linea):
    c = linea.split(",")          # split UNA sola vez
    return (c[0], c[1], c[2])     # (nombre, edad, ciudad/3ª columna)

datos = personas_sin_header.map(parsear)
datos.take(5)

[('John Doe', '30', 'Male'),
 ('Jane Smith', '25', 'Female'),
 ('David Johnson', '35', 'Male'),
 ('Emily Davis', '28', 'Female'),
 ('Michael Wilson', '40', 'Male')]

**4. ¿Por qué es más eficiente?**
- Se hace `split()` una vez por línea en lugar de tres → ~3 veces menos trabajo de CPU.
- Se recorre el RDD una sola vez y se obtiene una tupla con los tres campos, en lugar de generar tres RDDs que vuelven a leer el archivo cada uno.
- Si después necesitamos los campos por separado, podemos hacer `datos.map(lambda x: x[0])` sobre datos ya parseados (y opcionalmente `datos.cache()`).

## Ejercicio 7: Debugging
| # | Error | Por qué es un error | Corrección |
|---|---|---|---|
| 1 | No se obtiene `sc` | Se usa `sc.textFile` pero `sc` nunca se definió → `NameError` | `sc = spark.sparkContext` |
| 2 | `"./datos.csv"` | Sin `file://` Spark busca en HDFS y la ruta relativa puede fallar | `"file://" + os.path.abspath(...)` |
| 3 | `rdd.collect()` | Trae TODO el dataset a la memoria del driver → `OutOfMemoryError` en datos grandes | `take(n)` o `foreach` / `saveAsTextFile` |
| 4 | No se cierra la sesión | Quedan memoria, CPU y el puerto 4040 ocupados | `spark.stop()` |

> 📸 **Antes de ejecutar la siguiente celda**, si aún no tomaste capturas de la Spark UI (http://localhost:4040), hazlo ahora: esta celda cierra la sesión.

In [20]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ejercicio7").getOrCreate()
sc = spark.sparkContext                                        # Corrección 1

path = "file://" + os.path.abspath("./DATOS/persons.csv")     # Corrección 2 (datos.csv no existe; usamos persons.csv)
rdd = sc.textFile(path)

for linea in rdd.take(10):                                     # Corrección 3
    print(linea)

spark.stop()                                                   # Corrección 4

name,age,gender,salary
John Doe,30,Male,50000
Jane Smith,25,Female,45000
David Johnson,35,Male,60000
Emily Davis,28,Female,52000
Michael Wilson,40,Male,75000
Sarah Brown,32,Female,58000
Robert Lee,29,Male,51000
Lisa Garcia,27,Female,49000
James Martinez,38,Male,70000


---
# PROYECTO: Análisis de `stocks.txt`

> **Adaptación necesaria:** el enunciado describe `Símbolo,Fecha,Precio,Volumen`, pero el archivo real
> `stocks.txt` del repositorio contiene un inventario: `id,name,category,quantity,price` (30 registros).
> Para conservar la estructura del proyecto se usa esta equivalencia:
>
> | Enunciado | Columna real |
> |---|---|
> | símbolo | `category` (6 categorías) |
> | fecha | `name` (producto) |
> | precio | `price` |
> | volumen | `quantity` |
>
> Así, la tupla sigue siendo `(símbolo, fecha, precio, volumen)` y todo el código guía funciona igual.

## Paso 1: Configuración inicial y carga

In [21]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("AnalisisStocks") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()

sc = spark.sparkContext

path_relativo = "./DATOS/stocks.txt"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto
print(f"Leyendo desde: {path_final}")

rdd = sc.textFile(path_final)

print(rdd.take(5))

Leyendo desde: file:///home/jovyan/work/projects/DATOS/stocks.txt
['id,name,category,quantity,price', '1,iPhone,Electronics,10,899.99', '2,Macbook,Electronics,5,1299.99', '3,iPad,Electronics,15,499.99', '4,Samsung TV,Electronics,8,799.99']


## Paso 2: Limpieza y parseo

In [22]:
def parsear_linea(linea):
    try:
        partes = linea.split(",")          # un solo split
        simbolo = partes[2].strip()        # category  -> "símbolo"
        fecha = partes[1].strip()          # name      -> "fecha"
        precio = float(partes[4])          # price
        volumen = int(partes[3])           # quantity  -> "volumen"
        return (simbolo, fecha, precio, volumen)
    except:
        return None                        # el encabezado (o líneas corruptas) no se pueden convertir

rdd_parseado = rdd.map(parsear_linea).filter(lambda x: x is not None)

print(f"Líneas válidas: {rdd_parseado.count()}")
rdd_parseado.take(5)

Líneas válidas: 30


[('Electronics', 'iPhone', 899.99, 10),
 ('Electronics', 'Macbook', 1299.99, 5),
 ('Electronics', 'iPad', 499.99, 15),
 ('Electronics', 'Samsung TV', 799.99, 8),
 ('Electronics', 'LG TV', 699.99, 10)]

## Paso 3: Estadísticas por símbolo (máx, mín, promedio)

In [23]:
simbolo_precio = rdd_parseado.map(lambda x: (x[0], x[2]))

precio_max = simbolo_precio.reduceByKey(lambda a, b: max(a, b))
precio_min = simbolo_precio.reduceByKey(lambda a, b: min(a, b))

# aggregateByKey(valor_inicial, seqFunc, combFunc)
precio_promedio = simbolo_precio.aggregateByKey(
    (0.0, 0),                                          # (suma, contador)
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),   # sumar precio y contar 1
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])  # unir particiones
).mapValues(lambda x: x[0] / x[1])

print("Precio máximo por símbolo:")
for simbolo, precio in sorted(precio_max.collect()):
    print(f"{simbolo}: ${precio:.2f}")

print("\nPrecio mínimo por símbolo:")
for simbolo, precio in sorted(precio_min.collect()):
    print(f"{simbolo}: ${precio:.2f}")

print("\nPrecio promedio por símbolo:")
for simbolo, precio in sorted(precio_promedio.collect()):
    print(f"{simbolo}: ${precio:.2f}")

Precio máximo por símbolo:
Accessories: $29.99
Clothing: $199.99
Electronics: $1299.99
Food: $5.99
Furniture: $249.99
Sports: $49.99

Precio mínimo por símbolo:
Accessories: $24.99
Clothing: $14.99
Electronics: $129.99
Food: $0.25
Furniture: $79.99
Sports: $19.99

Precio promedio por símbolo:
Accessories: $27.49
Clothing: $99.28
Electronics: $586.66
Food: $2.30
Furniture: $141.99
Sports: $34.99


## Paso 4: Top 10 por volumen (aquí: productos con más unidades)

In [24]:
fecha_volumen = rdd_parseado.map(lambda x: (x[1], x[3]))

volumen_por_fecha = fecha_volumen.reduceByKey(lambda a, b: a + b)

top_fechas = volumen_por_fecha.sortBy(lambda x: x[1], ascending=False)

print("Top 10 con mayor volumen:")
for fecha, volumen in top_fechas.take(10):
    print(f"{fecha}: {volumen:,} unidades")

Top 10 con mayor volumen:
Bananas: 150 unidades
Oranges: 120 unidades
Apples: 100 unidades
Chicken Breast: 50 unidades
T-shirt: 50 unidades
Sneakers: 40 unidades
Backpack: 30 unidades
Nike Shoes: 30 unidades
Salmon Fillet: 30 unidades
Jeans: 30 unidades


## Paso 5: Volatilidad por símbolo

In [25]:
stats_por_simbolo = simbolo_precio.aggregateByKey(
    (float('-inf'), float('inf')),                              # (max, min)
    lambda acc, precio: (max(acc[0], precio), min(acc[1], precio)),
    lambda acc1, acc2: (max(acc1[0], acc2[0]), min(acc1[1], acc2[1]))
)

volatilidad = stats_por_simbolo.mapValues(
    lambda x: ((x[0] - x[1]) / x[1]) * 100
)

mas_volatiles = volatilidad.sortBy(lambda x: x[1], ascending=False).take(5)
mas_estables = volatilidad.sortBy(lambda x: x[1], ascending=True).take(5)

print("Símbolos más volátiles:")
for simbolo, vol in mas_volatiles:
    print(f"{simbolo}: {vol:.2f}%")

print("\nSímbolos más estables:")
for simbolo, vol in mas_estables:
    print(f"{simbolo}: {vol:.2f}%")

Símbolos más volátiles:
Food: 2296.00%
Clothing: 1234.16%
Electronics: 900.07%
Furniture: 212.53%
Sports: 150.08%

Símbolos más estables:
Accessories: 20.01%
Sports: 150.08%
Furniture: 212.53%
Electronics: 900.07%
Clothing: 1234.16%


## Paso 6: Guardar resultados y resumen ejecutivo
> Modificación mínima: `saveAsTextFile` falla si la carpeta ya existe, por eso se borra antes (permite re-ejecutar la celda).

In [26]:
import shutil
!mkdir -p RESULTADOS

output_path_volumen = "file://" + os.path.abspath("./RESULTADOS/volumen_por_fecha")
shutil.rmtree(os.path.abspath("./RESULTADOS/volumen_por_fecha"), ignore_errors=True)
volumen_por_fecha.saveAsTextFile(output_path_volumen)

output_path_stats = "file://" + os.path.abspath("./RESULTADOS/stats_por_simbolo")
shutil.rmtree(os.path.abspath("./RESULTADOS/stats_por_simbolo"), ignore_errors=True)
stats_por_simbolo.saveAsTextFile(output_path_stats)

!ls -R RESULTADOS

RESULTADOS:
stats_por_simbolo  volumen_por_fecha

RESULTADOS/stats_por_simbolo:
part-00000  part-00001	_SUCCESS

RESULTADOS/volumen_por_fecha:
part-00000  part-00001	_SUCCESS


In [27]:
total_registros = rdd_parseado.count()
simbolos_unicos = rdd_parseado.map(lambda x: x[0]).distinct().count()
simbolo_mayor_precio = precio_promedio.sortBy(lambda x: x[1], ascending=False).first()
dia_mayor_volumen = volumen_por_fecha.sortBy(lambda x: x[1], ascending=False).first()
simbolo_mas_volatil = volatilidad.sortBy(lambda x: x[1], ascending=False).first()

print("\n" + "="*60)
print("RESUMEN EJECUTIVO - ANÁLISIS DE STOCKS")
print("="*60)
print(f"\nTotal de registros procesados: {total_registros:,}")
print(f"Número de símbolos únicos: {simbolos_unicos}")
print(f"\nSímbolo con mayor precio promedio: {simbolo_mayor_precio[0]} (${simbolo_mayor_precio[1]:.2f})")
print(f"Día con mayor volumen: {dia_mayor_volumen[0]} ({dia_mayor_volumen[1]:,} acciones)")
print(f"Símbolo más volátil: {simbolo_mas_volatil[0]} ({simbolo_mas_volatil[1]:.2f}%)")
print(f"\nParticiones del RDD original: {rdd.getNumPartitions()}")


RESUMEN EJECUTIVO - ANÁLISIS DE STOCKS

Total de registros procesados: 30
Número de símbolos únicos: 6

Símbolo con mayor precio promedio: Electronics ($586.66)
Día con mayor volumen: Bananas (150 acciones)
Símbolo más volátil: Food (2296.00%)

Particiones del RDD original: 2


> 📸 **Ahora** abre http://localhost:4040 y captura las pestañas **Jobs**, **Stages**, **Executors** y **Environment** de la app *AnalisisStocks*. Después ejecuta la última celda.

In [28]:
spark.stop()
print("\n✅ Análisis completado. Sesión cerrada.")


✅ Análisis completado. Sesión cerrada.


---
# Preguntas de reflexión

**1. Eficiencia.** `split()` recorre todo el string y crea una lista nueva; hacerlo varias veces por línea multiplica el trabajo de CPU y la creación de objetos. Con 1 millón de líneas, pasar de 2–3 splits a 1 elimina 1–2 millones de operaciones; el ahorro es aproximadamente proporcional (la mitad o dos tercios del tiempo de parseo).

**2. Particionamiento.** Mi RDD leído con `textFile` tiene 2 particiones (ver salida del Paso 6); en cambio, el RDD de “Hola Spark” creado con `parallelize` tuvo 6, una por cada core disponible en el contenedor. Cada partición es una tarea que puede correr en paralelo en un core distinto. Con 1 sola partición todo se procesaría secuencialmente en un solo core, sin aprovechar el paralelismo; con demasiadas, el costo de gestionar tareas supera el beneficio.

**3. Lazy evaluation.** `map`, `filter`, `reduceByKey`, `sortBy`… solo construyen un plan (DAG). Se ejecutaron realmente al llamar acciones: `take`, `count`, `collect`, `first`, `saveAsTextFile`. Ventajas: Spark optimiza el plan completo, une transformaciones en una sola etapa (pipelining) y no calcula nada que no se use.

**4. Memoria.** `collect()` sobre 10 GB intentaría meter todo en la RAM del driver → `OutOfMemoryError` o `Py4JJavaError`. Se evita usando `take(n)`, agregaciones que reduzcan el tamaño (`count`, `reduceByKey`) o escribiendo el resultado con `saveAsTextFile`.

**5. Optimización (100x datos).** Aumentar particiones (`repartition`, `minPartitions` en `textFile`), cachear `rdd_parseado` con `.cache()` porque se reutiliza en varios pasos, calcular max/min/suma en una sola pasada con `aggregateByKey`, ajustar memoria de executors y pasar a DataFrames (optimizador Catalyst).

**6. HDFS vs local.** HDFS distribuye y replica los bloques en varios nodos: tolerancia a fallos (si un nodo cae se lee otra réplica), lectura en paralelo desde varios discos, *data locality* (el cómputo va donde está el dato) y capacidad para archivos más grandes que un disco. En local todo depende de una sola máquina.